# Pattern ⑥ 観測性・MLOps連携

## 概要

Difyアプリの観測性（Observability）をDatabricks MLflowで実現します。  
Dify v1.10.1以降、トレーシングプロバイダーに**Databricks**が追加され、UI設定のみでLLM/ツールトレースをMLflow Experimentに送信できます。

**アーキテクチャ:**

```
Dify アプリ → Databricks ネイティブプロバイダー → MLflow Experiment → Delta Sync (~15分) → UC Delta Table
```

このノートブックでは主に **方式A: Delta Sync（推奨）** をハンズオンで実施し、参考として **方式B: Zerobus OTEL Direct** の検証結果も紹介します。

## 2つの方式の比較

| 観点 | A: Delta Sync（推奨） | B: Zerobus OTEL Direct |
|------|---------------------|------------------------|
| **データ経路** | Dify → Databricksプロバイダー → MLflow Experiment → Delta Sync → UC Delta Table | Dify → MLflow SDK (set_destination) → OTEL protobuf → Zerobus `/api/2.0/otel/v1/traces` → UC OTEL Table |
| **コード変更** | 不要（UI設定のみ） | 必要（mlflow_trace.py変更 + docker-compose.override.yml） |
| **ディレイ** | ~15分（Delta Sync間隔） | リアルタイム |
| **アプリ識別** | `trace_location.mlflow_experiment.experiment_id` で識別可能 | UC OTELテーブルにexperiment_idカラムがなく、複数アプリの識別不可 |
| **方式Aとの併用** | — | 不可（set_destinationがset_experimentを上書き） |
| **Dify UI制御** | アプリごとに設定可能 | 環境変数のみ（UI制御不可） |
| **推奨度** | **推奨** | 参考（本番リアルタイム要件時） |

### 方式A推奨の理由

- コード変更不要で、Dify UIからアプリごとに設定可能
- Delta Syncテーブルの `experiment_id` で複数アプリを同一テーブル内で識別可能
- MLflow SDKのトレース検索 + AI Judgeで品質評価がそのまま使える
- SQL分析もダッシュボードも標準的なDatabricksワークフローで実現可能

## 前提条件

- `00_data_setup` ノートブック実行済み
- `02_http_api_integration` または `03_mcp_integration` 実行済み
- Difyで対象アプリにて会話を数回実施済み（トレースデータが必要）

---

## 方式A: Delta Sync（推奨）Hands-on

Dify → Databricksネイティブプロバイダー → MLflow Experiment → Delta Sync → UC Delta Table のフローを構築します。

In [ ]:
%pip install mlflow[databricks]>=3.9 databricks-agents -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./_config

### Step 1: MLflow Experiment作成

DifyアプリごとにMLflow Experimentを作成します。  
Experiment IDはDifyの監視設定で使用します。

In [ ]:
import mlflow

print(f"MLflow version: {mlflow.__version__}")

experiment_name = MLFLOW_EXPERIMENT_NAME  # config.yaml から取得
mlflow.set_tracking_uri("databricks")

try:
    exp = mlflow.get_experiment_by_name(experiment_name)
    if exp is not None:
        experiment_id = exp.experiment_id
        print(f"Experimentが見つかりました: {experiment_name}")
    else:
        experiment_id = mlflow.create_experiment(experiment_name)
        print(f"Experimentを作成しました: {experiment_name}")
except Exception as e:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"Experimentを新規作成しました: {experiment_name}")

print(f"Experiment ID: {experiment_id}")
print(f"\nDifyの監視設定でこのExperiment IDを使用してください。")

### Step 2: Dify UI設定

Difyアプリ側でトレーシングを有効化します。アプリごとに設定が必要です。

1. Difyで対象アプリを開く
2. 左メニュー **「監視（Monitoring）」** をクリック
3. **「設定（Settings）」** → プロバイダー一覧から **Databricks** を選択
4. 以下を入力:
   - **Host**: DatabricksワークスペースURL
   - **Token**: Personal Access Token
   - **Experiment ID**: 上記Step 1で取得したID
5. **「保存」** をクリック

> 設定後、Difyで会話を実施するとMLflow Experimentにトレースが記録されます。

### Step 3: Delta Sync有効化

MLflow ExperimentのトレースをUC Delta Tableに自動同期します。

1. Databricksワークスペースで **Machine Learning** → **Experiments** を開く
2. 対象のExperimentを選択
3. **「Traces」** タブを開く
4. **「Sync to Unity Catalog」** をクリック
5. 同期先のUCスキーマを指定:
   - Catalog: `yao_sl_st_catalog`
   - Schema: `dify_observability`
6. 有効化を確認

> Delta Syncは約15分間隔で実行されます。初回同期には少し時間がかかります。  
> 同期先テーブル: `{catalog}.{schema_obs}.{delta_sync_table}`

### Step 4: トレースの確認

MLflow SDKでトレースを取得し、スパン名で分類します。

| スパン名 | 内容 |
|---------|------|
| `message` | Agent/Chatbot メッセージ処理 |
| `mcp_sse_call_tool` | MCPツール呼び出し |
| `workflow` | Workflow実行 |
| `tool` | ビルトインツール |
| `llm` | LLM呼び出し |

In [ ]:
import mlflow
import json

traces = mlflow.search_traces(locations=[experiment_id], max_results=50)
print(f"✅ 記録済みトレース数: {len(traces)}")

# スパン名で分類
llm_traces = []       # LLM呼び出し（最終回答）
tool_traces = []      # ツール呼び出し（MCP等）
search_traces_list = []  # 検索（RAG）
workflow_traces = []  # Workflow
other_traces = []     # その他

for _, t in traces.iterrows():
    spans = t.get("spans", [])
    if not spans:
        other_traces.append(t)
        continue
    root_name = spans[0]["name"] if isinstance(spans, list) and len(spans) > 0 else "N/A"
    if "message" in root_name or "llm" in root_name:
        llm_traces.append(t)
    elif "mcp" in root_name or "tool" in root_name:
        tool_traces.append(t)
    elif "search" in root_name or "retriev" in root_name:
        search_traces_list.append(t)
    elif "workflow" in root_name:
        workflow_traces.append(t)
    else:
        other_traces.append(t)

print(f"\n--- スパン分類 ---")
print(f"LLM / Message:  {len(llm_traces)}")
print(f"Tool / MCP:     {len(tool_traces)}")
print(f"Search / RAG:   {len(search_traces_list)}")
print(f"Workflow:       {len(workflow_traces)}")
print(f"Other:          {len(other_traces)}")

### Step 5: 品質評価（AI Judge）

MLflow 3のAI Judge機能で、トレースに対して自動品質評価を行います。

| 評価 | Scorer | 評価内容 |
|------|--------|--------|
| Safety | `Safety()` | 有害・危険な内容がないか |
| Guidelines | `Guidelines()` | 自然な日本語、ハルシネーションなし |

In [ ]:
from mlflow.genai.scorers import Guidelines, Safety
import mlflow.genai
import pandas as pd

# トレースから評価データセットを構築
eval_data = []
for _, t in traces.iterrows():
    req = t.get("request")
    res = t.get("response")
    if req and res:
        eval_data.append({
            "request": str(req)[:500],
            "response": str(res)[:500]
        })

print(f"評価対象トレース数: {len(eval_data)}")

if eval_data:
    eval_df = pd.DataFrame(eval_data)

    with mlflow.start_run(run_name="dify_quality_eval"):
        results = mlflow.genai.evaluate(
            data=eval_df,
            scorers=[
                Safety(),
                Guidelines(
                    name="japanese_quality",
                    guidelines="回答は自然な日本語で書かれており、ハルシネーションが含まれていない"
                ),
            ]
        )
        print("\n--- 評価結果 ---")
        display(results.tables["eval_results"])
else:
    print("評価対象のトレースがありません。Difyで会話を実施してから再実行してください。")

### Step 6: SQL分析（Delta Syncテーブル）

Delta SyncでUCに同期されたトレースデータをSQLで直接分析します。  
テーブル: `{catalog}.{schema_obs}.{delta_sync_table}`

> `trace_location.mlflow_experiment.experiment_id` でアプリごとのトレースを識別できます。

In [ ]:
# Delta Syncテーブル名を設定（Delta Sync UIで指定したテーブル名に合わせてください）
delta_sync_table = f"{catalog}.dify_observability.mlflow_trace_log_delta_sync"  # 例
print(f"Delta Sync テーブル: {delta_sync_table}")

In [ ]:
# アプリ（Experiment）別トレース集計
display(spark.sql("""
SELECT 
    trace_location.mlflow_experiment.experiment_id,
    spans[0].name as root_span,
    count(*) as trace_count,
    avg(execution_duration_ms) as avg_duration_ms
FROM {delta_sync_table}
GROUP BY 1, 2
ORDER BY trace_count DESC
"""))

In [ ]:
# 最新トレース一覧（レイテンシ・スパンタイプ）
display(spark.sql("""
SELECT
    spans[0].name as span_type,
    tags['mlflow.traceName'] as trace_name,
    execution_duration_ms,
    request_time
FROM {delta_sync_table}
ORDER BY request_time DESC
LIMIT 20
"""))

In [ ]:
# レイテンシ分布（スパンタイプ別）
display(spark.sql("""
SELECT 
    spans[0].name as span_type,
    count(*) as count,
    round(avg(execution_duration_ms), 0) as avg_ms,
    round(min(execution_duration_ms), 0) as min_ms,
    round(max(execution_duration_ms), 0) as max_ms,
    round(percentile(execution_duration_ms, 0.5), 0) as p50_ms,
    round(percentile(execution_duration_ms, 0.95), 0) as p95_ms
FROM {delta_sync_table}
GROUP BY 1
ORDER BY count DESC
"""))

### Step 7: モニタリングダッシュボード

時系列の集計クエリを使って、リクエスト数・レイテンシ・エラー率などをモニタリングできます。  
このクエリをDatabricks AI/BI Dashboardに登録すれば、リアルタイムの監視ダッシュボードが作成できます。  
また、Databricks Alertと組み合わせることでエラー率閾値超過時の通知も可能です。

In [ ]:
# 時間別集計（モニタリングダッシュボード用）
display(spark.sql("""
SELECT
    date_trunc('HOUR', request_time) as hour,
    count(*) as request_count,
    avg(execution_duration_ms) as avg_latency_ms,
    max(execution_duration_ms) as max_latency_ms,
    count(CASE WHEN state = 'OK' THEN 1 END) as success_count,
    count(CASE WHEN state != 'OK' THEN 1 END) as error_count
FROM {delta_sync_table}
GROUP BY 1
ORDER BY 1 DESC
"""))

---

## 方式B: Zerobus OTEL Direct（参考）

### 概要

MLflow SDKの `set_destination` を使い、OTEL protobufでDatabricks ZerobusのOTLPエンドポイントに直接トレースを送信する方式です。  
リアルタイムでUC OTEL Tableに書き込まれますが、現時点では制約が多く参考情報に留めます。

### データ経路

```
Dify → MLflow SDK (mlflow_trace.py変更) → OTEL protobuf → Zerobus /api/2.0/otel/v1/traces → UC OTEL Table
```

### 必要なコード変更

1. **`mlflow_trace.py`**: `set_destination` の追加、`dify_graph` importの修正
2. **`docker-compose.override.yml`**: `MLFLOW_TRACING_DESTINATION` 環境変数の設定

コード変更はDify forkブランチ `mlflow-zerobus-otel` に保存してあります。

### 制約事項

| 制約 | 説明 |
|------|------|
| **現行スキーマでexperiment_idなし** | 同一UCスキーマ内で複数アプリを識別できない |
| **set_destinationがset_experimentを上書き** | 方式A（Delta Sync）と併用不可 |
| **Dify UIで制御不可** | 環境変数のみで設定（アプリ単位の切り替え不可） |

### 今後のロードマップ

- **MLflow 3.11.0**: 次期スキーマでtable prefixサポートが追加予定。これにより複数Experimentの識別が可能になる見込み
- 次期バージョン（MLflow 3.11.0+）サポート後は、本番環境でのリアルタイムトレーシングに有効

### 方式Bが適切なケース

- 本番環境でリアルタイムのトレース収集が必要な場合
- カスタムOTELパイプラインを構築する場合
- Delta Syncの15分ディレイが許容できない場合

---

## まとめ

Difyアプリの観測性をDatabricksで実現する2つの方式を検証しました。

| 方式 | 用途 | 状況 |
|------|------|------|
| **A: Delta Sync（推奨）** | 日常の品質監視・SQL分析 | UI設定のみで利用可能 |
| **B: Zerobus OTEL** | リアルタイム監視 | 参考（V2待ち） |

**推奨ワークフロー:**

1. **方式A** でDifyアプリごとにMLflow Experimentを作成・Delta Syncを有効化
2. MLflow SDKでトレース取得 → AI Judgeで品質評価
3. Delta SyncテーブルでSQL分析 → Databricks Dashboardで可視化
4. Databricks Alertでエラー率・レイテンシの閾値監視

**検証済みExperiment:**

| アプリ | Experiment ID |
|------|---------------|
| A-1 MCP Agent | {A-1_experiment_id} |
| A-2 UC Workflow | {A-2_experiment_id} |